Imports

In [ ]:
import json
from pathlib import Path
import geoai
import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.geometry import Point
#from geotessera import GeoTesseraZarr
import ee
from obstore.store import S3Store
from pyproj import Transformer
from tqdm import tqdm
import zarr
import s3fs
import sys
from zarr.storage import ObjectStore

Load the modeling csv 

In [2]:
output_dir = Path("output")
# Find subfolders in output folder to choose input

subfolders = sorted([f for f in output_dir.iterdir() if f.is_dir()])

if not subfolders:
    raise FileNotFoundError(f"No subdirectories found inside '{output_dir}'.")

if len(subfolders) == 1:
    selected_folder = subfolders[0]
    print(f"Only one folder found. Automatically selected: {selected_folder.name}.")

else:
    print("Available folders in output directory:")
    for idx, folder in enumerate(subfolders, start=1):
        print(f"[{idx}] {folder.name}")

# Get user input
    user_choice = input(f"\nEnter folder number (1-{len(subfolders)}): ").strip()

    if not user_choice.isdigit() or not (1 <= int(user_choice) <= len(subfolders)):
        raise ValueError(
            f"Invalid selection '{user_choice}'. Expected number between 1 and {len(subfolders)}."
        )

    selected_folder = subfolders[int(user_choice) - 1]
    print(f"\n --> Selected folder: {selected_folder.name}")


# Locate modeling file
modeling_files = sorted(selected_folder.glob("*_modeling*.csv"))

# Pick the file
modeling_file = modeling_files[0]

if not modeling_file:
    raise FileNotFoundError(f"No '*_modeling*.csv' file found in {selected_folder.name}.")

print(f"Loading dataset: {modeling_file.name}")

df_master = pd.read_csv(modeling_file, sep=";")
df_master["Year"] = df_master["Year"].astype(int)

# Filter to TESSERA availability window (2017 onwards)
df_tessera_base = df_master[df_master["Year"] >= 2017].copy().reset_index(
    drop=True
)

Available folders in output directory:
[1] france_sbt_bbch39_2017-01-01_2022-12-31
[2] germany_barw_bbch51_2017-01-01_2025-12-31
[3] germany_whbw_bbch0_2017-01-01_2020-12-31

 --> Selected folder: germany_barw_bbch51_2017-01-01_2025-12-31
Loading dataset: curated_for_modeling.csv


Extract the TESSERA embeddings

In [3]:
gt = GeoTesseraZarr()
print(f"Connected to TESSERA Cloud Zarr store.")
print(f"Sampling {len(df_tessera_base)} points across years {sorted(df_tessera_base['Year'].unique())}")

tessera_cols = [f"tessera_{i}" for i in range(128)]
extracted_year_dfs = []
unique_extract_years = sorted(df_tessera_base["Year"].unique())

for y in unique_extract_years:
    df_year = df_tessera_base[df_tessera_base["Year"] == y].copy().reset_index(drop=True)
    print(f"\n--- Streaming TESSERA embeddings for year {y} ({len(df_year)} points) ---")
    
    # Coordinates must be (longitude, latitude) pairs
    coords = list(zip(df_year["Longitude"], df_year["Latitude"]))
    
    # Stream in batches of 50 to keep memory minimal (< 1 MB)
    batch_size = 50
    batch_embs = []
    for start_idx in range(0, len(coords), batch_size):
        batch_coords = coords[start_idx : start_idx + batch_size]
        end_idx = min(start_idx + batch_size, len(coords))
        print(f"  Streaming points {start_idx + 1} to {end_idx} of {len(coords)}...")
        
        # Streams only the exact 128 pixel channels per point
        arr = gt.sample_points(batch_coords, year=int(y))
        batch_embs.append(arr)
        
    year_arr = np.vstack(batch_embs)
    
    # Attach embedding columns
    df_emb = pd.DataFrame(year_arr, columns=tessera_cols)
    df_year_combined = pd.concat([df_year, df_emb], axis=1)
    extracted_year_dfs.append(df_year_combined)


Connected to TESSERA Cloud Zarr store.
Sampling 4942 points across years [np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

--- Streaming TESSERA embeddings for year 2017 (621 points) ---
  Streaming points 1 to 50 of 621...
  Streaming points 51 to 100 of 621...
  Streaming points 101 to 150 of 621...
  Streaming points 151 to 200 of 621...
  Streaming points 201 to 250 of 621...
  Streaming points 251 to 300 of 621...
  Streaming points 301 to 350 of 621...
  Streaming points 351 to 400 of 621...
  Streaming points 401 to 450 of 621...
  Streaming points 451 to 500 of 621...
  Streaming points 501 to 550 of 621...
  Streaming points 551 to 600 of 621...
  Streaming points 601 to 621 of 621...

--- Streaming TESSERA embeddings for year 2018 (598 points) ---
  Streaming points 1 to 50 of 598...
  Streaming points 51 to 100 of 598...
  Streaming points 101 to 150 of 598...
  Streaming points 1

Combine the embeddings with the previous csv 

In [4]:
df_with_tessera = pd.concat(extracted_year_dfs, ignore_index=True)

valid_count = df_with_tessera["tessera_0"].notna().sum()
print(f"\nExtraction complete! Valid embeddings: {valid_count} / {len(df_with_tessera)}")

# Save enriched dataset to disk so you only ever have to run this once
enriched_file = selected_folder / f"{modeling_file.stem}_with_tessera_embeddings.csv"
df_with_tessera.to_csv(enriched_file, sep=";", index=False)
print(f"Saved enriched master file to: {enriched_file.resolve()}")

display(
    df_with_tessera[
        [
            "Latitude",
            "Longitude",
            "Year",
            "DOY",
            "tessera_0",
            "tessera_1",
            "tessera_64",
            "tessera_127",
        ]
    ].head()
)


Extraction complete! Valid embeddings: 4658 / 4942
Saved enriched master file to: C:\Users\jimpa\Desktop\Thesis\Notebooks\output\germany_barw_bbch51_2017-01-01_2025-12-31\curated_for_modeling_with_tessera_embeddings.csv


,Latitude,Longitude,Year,DOY,tessera_0,tessera_1,tessera_64,tessera_127
0,47.7333,9.06667,2017,140,-0.595965,-0.404405,-0.404405,0.127707
1,47.7500,12.11670,2017,134,NaN,NaN,NaN,NaN
2,47.7667,9.31667,2017,131,0.060531,-0.201772,-0.766732,0.726378
3,47.7667,9.53333,2017,129,0.132874,-0.708661,-0.686516,0.487205
4,47.8167,7.63333,2017,123,NaN,NaN,NaN,NaN


# Extract AlphaEarth embeddings

Use initial df to get coordinates & dates - Extract AlphaEarth embeddings

In [ ]:
# Initialize Earth Engine
try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize()

# AlphaEarth covers 2017 to 2025
available_years = [y for y in sorted(df_master["Year"].unique()) if 2017 <= y <= 2025]
print(f"Extracting AlphaEarth for years: {available_years}")

alphaearth_ic = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
alphaearth_bands = [f"A{i:02d}" for i in range(64)]

extracted_dfs = []

for year in available_years:
  df_year = df_master[df_master["Year"] == year].copy()
  if df_year.empty:
    continue

  print(f"\nProcessing year {year} ({len(df_year)} sites)...")

  start_date = f"{year}-01-01"
  end_date = f"{year + 1}-01-01"

  # Filter collection by date and select all 64 embedding bands
  annual_mosaic = (
      alphaearth_ic.filterDate(start_date, end_date)
      .select(alphaearth_bands)
      .mosaic()
  )

  # Convert points to an ee.FeatureCollection with a 10m buffer to align with pixel grid
  features = []
  for idx, row in df_year.iterrows():
    pt = ee.Geometry.Point([row["Longitude"], row["Latitude"]])
    feat = ee.Feature(pt, {"row_id": int(idx)})
    features.append(feat)

  pts_fc = ee.FeatureCollection(features)

  # Chunk in batches of 500 points to avoid Earth Engine memory/timeout issues
  batch_size = 500
  feature_list = pts_fc.toList(pts_fc.size())
  num_batches = int(np.ceil(len(features) / batch_size))

  sampled_records = []
  for b in range(num_batches):
    sub_fc = ee.FeatureCollection(
        feature_list.slice(b * batch_size, (b + 1) * batch_size)
    )

    sampled_sub = annual_mosaic.reduceRegions(
        collection=sub_fc, reducer=ee.Reducer.first(), scale=10
    )

    data = sampled_sub.getInfo()["features"]
    for item in data:
      sampled_records.append(item["properties"])

  df_sampled = pd.DataFrame(sampled_records)

  if df_sampled.empty:
    print(f"Warning: No embeddings extracted for {year}.")
    continue

  df_sampled = df_sampled.set_index("row_id")

  # Rename A00..A63 -> alphaearth_0..alphaearth_63
  rename_dict = {f"A{i:02d}": f"alphaearth_{i}" for i in range(64)}
  df_sampled = df_sampled.rename(columns=rename_dict)

  alpha_cols = [f"alphaearth_{i}" for i in range(64)]
  # Keep only present columns
  valid_alpha_cols = [c for c in alpha_cols if c in df_sampled.columns]

  df_merged_year = df_year.join(df_sampled[valid_alpha_cols])
  extracted_dfs.append(df_merged_year)
  print(f"Successfully extracted {len(df_merged_year)} records for {year}.")

# -------------------------------------------------------------------------
# 4. Save Final CSV
# -------------------------------------------------------------------------
if extracted_dfs:
  df_alphaearth_final = pd.concat(extracted_dfs, axis=0).sort_values(
      ["Year", "DOY"]
  )
  save_path = (
      selected_folder / "curated_for_modeling_with_alphaearth_embeddings.csv"
  )
  df_alphaearth_final.to_csv(save_path, sep=";", index=False)
  print(
      f"\nComplete! Saved {len(df_alphaearth_final)} rows with AlphaEarth"
      " embeddings to:"
  )
  print(save_path.resolve())
else:
  print("Extraction returned no records.")

Extracting AlphaEarth for years: [np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Processing year 2017 (621 sites)...
Successfully extracted 621 records for 2017.

Processing year 2018 (598 sites)...
Successfully extracted 598 records for 2018.

Processing year 2019 (592 sites)...
Successfully extracted 592 records for 2019.

Processing year 2020 (579 sites)...
Successfully extracted 579 records for 2020.

Processing year 2021 (543 sites)...
Successfully extracted 543 records for 2021.

Processing year 2022 (529 sites)...
Successfully extracted 529 records for 2022.

Processing year 2023 (511 sites)...
Successfully extracted 511 records for 2023.

Processing year 2024 (488 sites)...
Successfully extracted 488 records for 2024.

Complete! Saved 4461 rows with AlphaEarth embeddings to:
C:\Users\jimpa\Desktop\Thesis\Notebooks\output\germany_barw_bbch51_2017-01-01_2025-12-31\curated_for_modeling_with_alphaeart

OLMOEarth embeddings extraction

In [3]:
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np
from obstore.store import S3Store
import pandas as pd
from pyproj import Transformer
from tqdm import tqdm
import zarr
from zarr.storage import ObjectStore

# -------------------------------------------------------------------------
# 1. Connect to Source Cooperative Cloud Store
# -------------------------------------------------------------------------
zarr.config.set({"array.sharding_coalesce_max_gap_bytes": 0})

s3 = S3Store.from_url(
    "s3://olmoearth/olmoearth-v1-3-embeddings/embeddings.zarr",
    endpoint="https://data.source.coop",
    skip_signature=True,
)
zarr_store = ObjectStore(s3, read_only=True)
root = zarr.open_group(store=zarr_store, mode="r")

print("Connected to OlmoEarth v1.3 Zarr archive successfully!")


def dequantize(q):
  q = np.asarray(q)
  v = np.where(q == -128, np.nan, q).astype("float32")
  return (np.abs(v) / 127.5) ** 2.0 * np.sign(v)


# -------------------------------------------------------------------------
# 2. Input Dataset Configuration
# -------------------------------------------------------------------------
print(f"Loaded {len(df_tessera_base)} observations from {modeling_file.name}")
N = len(df_tessera_base)
dim = 128

# -------------------------------------------------------------------------
# 3. Cache UTM Zone Transformers & Affine Matrices
# -------------------------------------------------------------------------
print("Pre-caching zone affine transforms...")
utm_meta = {}
for zone_num in [32, 33]:
  z_name = f"utm{zone_num:02d}"
  if z_name in root:
    zone = root[z_name]
    a, _, c, _, e, f = zone.attrs["spatial:transform"]
    utm_meta[zone_num] = {
        "transformer": Transformer.from_crs(
            "EPSG:4326", zone.attrs["proj:code"], always_xy=True
        ),
        "affine": (a, c, e, f),
        "time": np.array(zone["time"]),
        "emb_ds": zone["embeddings"],
    }

# -------------------------------------------------------------------------
# 4. Resolve Affine Pixel Indices
# -------------------------------------------------------------------------
print("Resolving pixel indices via spatial transform...")
query_tasks = []

for idx, row in df_tessera_base.iterrows():
  lat = float(row["Latitude"])
  lon = float(row["Longitude"])
  yr = int(row["Year"])

  zone_num = int((lon + 180) // 6) + 1

  if zone_num not in utm_meta:
    query_tasks.append(None)
    continue

  meta = utm_meta[zone_num]
  if yr not in meta["time"]:
    query_tasks.append(None)
    continue

  time_idx = int(np.where(meta["time"] == yr)[0][0])

  # Accurate WGS84 -> UTM transformation
  ux, uy = meta["transformer"].transform(lon, lat)

  # Exact affine mapping to 10m raster indices
  a, c, e, f = meta["affine"]
  col_idx = int((ux - c) / a)
  row_idx = int((uy - f) / e)

  query_tasks.append((zone_num, time_idx, row_idx, col_idx))


# -------------------------------------------------------------------------
# 5. Concurrent Point Extraction (128 Dimensions)
# -------------------------------------------------------------------------
def fetch_128d(task):
  if task is None:
    return np.full(dim, np.nan, dtype=np.float32)
  zone_num, time_idx, row_idx, col_idx = task
  try:
    ds = utm_meta[zone_num]["emb_ds"]
    q_vec = ds[time_idx, :dim, row_idx, col_idx]
    return dequantize(q_vec)
  except Exception:
    return np.full(dim, np.nan, dtype=np.float32)


print(f"Streaming 128-d OlmoEarth embeddings across {N} observations...")
with ThreadPoolExecutor(max_workers=20) as executor:
  results = list(
      tqdm(
          executor.map(fetch_128d, query_tasks),
          total=N,
          desc="OlmoEarth 128d",
      )
  )

olmo_matrix = np.vstack(results)

# Diagnostics
nan_count = np.isnan(olmo_matrix).sum()
print(f"NaN elements before median imputation: {nan_count}")

if np.isnan(olmo_matrix).any():
  col_medians = np.nanmedian(olmo_matrix, axis=0)
  col_medians = np.nan_to_num(col_medians, nan=0.0)
  inds = np.where(np.isnan(olmo_matrix))
  olmo_matrix[inds] = np.take(col_medians, inds[1])

print(f"Final matrix norm mean: {np.linalg.norm(olmo_matrix, axis=1).mean():.2f}")
print(f"Matrix shape: {olmo_matrix.shape}")

# -------------------------------------------------------------------------
# 6. Save Curated Modeling File
# -------------------------------------------------------------------------
olmo_cols = [f"olmo_{i}" for i in range(dim)]
df_olmo = pd.DataFrame(
    olmo_matrix, columns=olmo_cols, index=df_tessera_base.index
)

base_cols = [c for c in df_tessera_base.columns if not c.startswith("presto_")]
df_final = pd.concat([df_tessera_base[base_cols], df_olmo], axis=1)

output_path = output_dir / "curated_for_modeling_with_olmo_embeddings.csv"
df_final.to_csv(output_path, sep=";", index=False)

print(f"\nSaved {output_path.name} with {dim} OlmoEarth dimensions!")
print(f"Path: {output_path.resolve()}")

Connected to OlmoEarth v1.3 Zarr archive successfully!
Loaded 4942 observations from curated_for_modeling.csv
Pre-caching zone affine transforms...
Resolving pixel indices via spatial transform...
Streaming 128-d OlmoEarth embeddings across 4942 observations...


OlmoEarth 128d: 100%|██████████| 4942/4942 [30:08<00:00,  2.73it/s]  


NaN elements before median imputation: 6400
Final matrix norm mean: 3.01
Matrix shape: (4942, 128)

Saved curated_for_modeling_with_olmo_embeddings.csv with 128 OlmoEarth dimensions!
Path: C:\Users\jimpa\Desktop\Thesis\Notebooks\output\curated_for_modeling_with_olmo_embeddings.csv
